In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/spaceship-titanic/sample_submission.csv
/kaggle/input/spaceship-titanic/train.csv
/kaggle/input/spaceship-titanic/test.csv


In [3]:
train_data = pd.read_csv("/kaggle/input/spaceship-titanic/train.csv")
test_data = pd.read_csv("/kaggle/input/spaceship-titanic/test.csv")

In [4]:
#change columns: 



In [5]:
train_data.isna().sum(axis=0)

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [6]:
train_data

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,9276_01,Europa,False,A/98/P,55 Cancri e,41.0,True,0.0,6819.0,0.0,1643.0,74.0,Gravior Noxnuther,False
8689,9278_01,Earth,True,G/1499/S,PSO J318.5-22,18.0,False,0.0,0.0,0.0,0.0,0.0,Kurta Mondalley,False
8690,9279_01,Earth,False,G/1500/S,TRAPPIST-1e,26.0,False,0.0,0.0,1872.0,1.0,0.0,Fayey Connon,True
8691,9280_01,Europa,False,E/608/S,55 Cancri e,32.0,False,0.0,1049.0,0.0,353.0,3235.0,Celeon Hontichre,False


In [7]:
train_data[["Deck", "CabinNum", "Side"]]= train_data["Cabin"].str.split("/", expand=True)
test_data[["Deck", "CabinNum", "Side"]]= train_data["Cabin"].str.split("/", expand=True)

train_data['CabinNum'] = pd.to_numeric(train_data['CabinNum'], errors='coerce')
test_data["CabinNum"] = pd.to_numeric(test_data["CabinNum"], errors='coerce')


In [8]:
train_data['Group'] = train_data['PassengerId'].str.split('_').str[0]
test_data['Group'] = test_data['PassengerId'].str.split('_').str[0]

train_data['GroupSize'] = train_data.groupby('Group')['Group'].transform('count')
test_data['GroupSize'] = test_data.groupby('Group')['Group'].transform('count')

In [9]:
spending = ['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']
train_data['TotalSpent'] = train_data[spending].sum(axis=1)
test_data['TotalSpent'] = test_data[spending].sum(axis=1)

In [10]:
train_data['CryoSleep'] = train_data['CryoSleep'].astype(float)
test_data['CryoSleep'] = test_data['CryoSleep'].astype(float)
train_data['VIP'] = train_data['VIP'].astype(float)
test_data['VIP'] = test_data['VIP'].astype(float)

In [11]:
spending = ['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']

for col in spending:
    train_data[col] = train_data[col].fillna(0)
    test_data[col] = test_data[col].fillna(0)

In [12]:
train_data['Age'] = train_data['Age'].fillna(train_data['Age'].median())
test_data['Age'] = train_data['Age'].fillna(test_data['Age'].median())

In [13]:
train_data['VIP'] = train_data['VIP'].fillna(False).astype(int)
test_data['VIP'] = test_data['VIP'].fillna(False).astype(int)

In [14]:
train_data['CryoSleep'] = [1 if True else 0 for i in train_data['CryoSleep']]
test_data['CryoSleep'] = [1 if True else 0 for i in test_data['CryoSleep']]

In [15]:
train_data.loc[(train_data[spending].sum(axis=1) == 0) & (train_data['CryoSleep'].isna()), 'CryoSleep'] = 1
test_data.loc[(test_data[spending].sum(axis=1) == 0) & (test_data['CryoSleep'].isna()), 'CryoSleep'] = 1

In [16]:
train_data['CryoSleep'] = train_data['CryoSleep'].fillna(0).astype(int)
test_data['CryoSleep'] = test_data['CryoSleep'].fillna(0).astype(int)

In [18]:
y = train_data['Transported'].map({True:1, False:0})

In [17]:
for col in ['Deck', 'Side']:
    train_data[col] = train_data[col].fillna(train_data[col].mode()[0])
    test_data[col] = test_data[col].fillna(test_data[col].mode()[0])

train_data['CabinNum'] = train_data['CabinNum'].fillna(train_data['CabinNum'].median())
test_data['CabinNum'] = test_data['CabinNum'].fillna(test_data['CabinNum'].median())

In [23]:
cat_features = ['HomePlanet', 'Destination']

for col in cat_features:
    train_data[col] = train_data[col].fillna(train_data[col].mode()[0])
    test_data[col] = test_data[col].fillna(test_data[col].mode()[0])

In [24]:
drop_columns = ['PassengerId', 'Name', 'Cabin', 'Group', 'Transported']

X = train_data.drop(columns=drop_columns)

X.loc[500:530]

,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Deck,CabinNum,Side,GroupSize,TotalSpent
500,Europa,1,55 Cancri e,36.0,0,0.0,0.0,0.0,0.0,0.0,C,18.0,P,5,0.0
501,Earth,1,55 Cancri e,30.0,0,0.0,0.0,0.0,0.0,0.0,C,18.0,P,5,0.0
502,Europa,1,TRAPPIST-1e,49.0,0,0.0,0.0,0.0,0.0,0.0,C,18.0,P,5,0.0
503,Europa,1,TRAPPIST-1e,38.0,0,0.0,1793.0,479.0,9.0,4987.0,C,18.0,P,5,7268.0
504,Mars,1,TRAPPIST-1e,31.0,0,0.0,0.0,0.0,0.0,0.0,F,95.0,S,2,0.0
505,Earth,1,TRAPPIST-1e,17.0,0,1471.0,0.0,0.0,45.0,16.0,F,95.0,S,2,1532.0
506,Earth,1,TRAPPIST-1e,26.0,0,9.0,0.0,0.0,0.0,878.0,G,84.0,P,1,887.0
507,Earth,1,TRAPPIST-1e,23.0,0,59.0,0.0,0.0,681.0,0.0,F,111.0,P,1,740.0
508,Earth,1,55 Cancri e,24.0,0,930.0,691.0,838.0,0.0,0.0,F,96.0,S,1,2459.0
509,Earth,1,TRAPPIST-1e,23.0,0,0.0,15.0,831.0,16.0,3.0,F,97.0,S,1,865.0


In [29]:
from sklearn.model_selection import train_test_split

X_train, x_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [30]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from catboost import CatBoostClassifier

cat_features = ['HomePlanet', 'Destination', 'Deck', 'Side']

cat_model = CatBoostClassifier(
    depth=6,
    learning_rate=0.05,
    iterations=1200,
    l2_leaf_reg=3,
    loss_function='Logloss',
    eval_metric='Accuracy',
    random_state=42,
    verbose=0,
    cat_features=cat_features  
)

cat_model.fit(X_train, y_train)
cat_pred = cat_model.predict(x_val)

cat_acc = accuracy_score(y_val, cat_pred)
print("CatBoost Accuracy:", cat_acc)

print("\nClassification Report (CatBoost):")
print(classification_report(y_val, cat_pred))

print("Confusion Matrix (CatBoost):")
print(confusion_matrix(y_val, cat_pred))

CatBoost Accuracy: 0.8159861989649224

Classification Report (CatBoost):
              precision    recall  f1-score   support

           0       0.82      0.81      0.81       863
           1       0.82      0.82      0.82       876

    accuracy                           0.82      1739
   macro avg       0.82      0.82      0.82      1739
weighted avg       0.82      0.82      0.82      1739

Confusion Matrix (CatBoost):
[[702 161]
 [159 717]]
